# 🎨 IndusCraft - Step 1: Dataset Preparation & Hugging Face Hub Push

This notebook ingests, cleans, deduplicates, captions, and packages Indian traditional craft imagery into a Hugging Face dataset ready for cloud GPU training.

In [1]:
# 1. Environment & Setup
import sys
from pathlib import Path

# Ensure project root is on Python path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(project_root))

from data_pipeline.acquisition import ensure_directories, ingest_local_directory, RAW_METADATA_PATH
from data_pipeline.cleaning import run_cleaning_pipeline
from data_pipeline.captioning import process_and_caption_dataset
from data_pipeline.hf_uploader import prepare_and_push_to_hf
from data_pipeline.config import CRAFT_METADATA

ensure_directories()
print("Workspace directories initialized.")

2026-08-27 12:28:54,338 - INFO - NumExpr defaulting to 12 threads.
2026-08-27 12:28:55,852 - INFO - All project directories verified/created successfully.


Workspace directories initialized.


## 📥 2. Ingest Raw Images
Specify the path to your raw images directory and select the craft category.

In [2]:
# Example: Ingest raw images
# Change raw_source_folder to your actual image directory
raw_source_folder = str(project_root / "data" / "sample_raw")
craft_label = "chikankari"  # Options: chikankari, phulkari, kalamkari, ajrakh, bandhani, kantha, paithani, ikat, madhubani, warli

if Path(raw_source_folder).exists():
    records = ingest_local_directory(raw_source_folder, craft_label)
    print(f"Ingested {len(records)} raw image records.")
else:
    print(f"Source directory '{raw_source_folder}' does not exist yet. Please place your raw images in {raw_source_folder}.")

Source directory 'c:\Users\HP\projects\diffusion_model\data\sample_raw' does not exist yet. Please place your raw images in c:\Users\HP\projects\diffusion_model\data\sample_raw.


## 🧹 3. Clean, Filter & Deduplicate Images
Filters corrupt files, enforces minimum 512x512 resolution, and removes near-duplicates via pHash.

In [ ]:
import json

if RAW_METADATA_PATH.exists():
    raw_records = []
    with open(RAW_METADATA_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                raw_records.append(json.loads(line))
    
    cleaned_records = run_cleaning_pipeline(raw_records)
    print(f"Cleaned records count: {len(cleaned_records)}")
else:
    print("No raw metadata found. Complete ingestion step first.")

## 📝 4. Auto-Caption & Normalize Prompts
Runs VLM captioning and normalizes craft metadata into training captions.

In [ ]:
# Set vlm_model to None for rule-based metadata builder or specify a HF VLM model (e.g. 'Salesforce/blip2-opt-2.7b')
vlm_model = None

if 'cleaned_records' in locals() and cleaned_records:
    train_recs, val_recs = process_and_caption_dataset(
        cleaned_records,
        vlm_model_name=vlm_model,
        val_split_ratio=0.1
    )
    print(f"Train samples: {len(train_recs)} | Val samples: {len(val_recs)}")
    
    # Preview sample caption
    if train_recs:
        print("\n--- Sample Caption ---")
        print(train_recs[0]["text"])

## 📤 5. Push Dataset to Hugging Face Hub
Package the dataset and upload to your Hugging Face Hub account for seamless cloud training.

In [ ]:
hf_repo_id = "your-hf-username/induscraft-dataset"  # Replace with your HF username and dataset name
hf_token = None  # Add your HF Write Token or run `huggingface-cli login` in terminal

if hf_repo_id != "your-hf-username/induscraft-dataset":
    prepare_and_push_to_hf(repo_id=hf_repo_id, token=hf_token, private=False)
    print(f"Dataset successfully published at https://huggingface.co/datasets/{hf_repo_id}")
else:
    print("Please set `hf_repo_id` to your Hugging Face username/dataset-name before pushing.")